<a href="https://colab.research.google.com/github/officiallong/BUS4-118S-build-ai-agents-and-chatbots-with-langgraph-2021112/blob/main/Working_Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:

import csv
from pathlib import Path
from collections import defaultdict

# PART 1: LOADING MARKETING DATA CSV

CSV_PATH = Path("/content/marketing_data.csv")

# If the file is not already attached, ask you to upload it.
if not CSV_PATH.exists():
    from google.colab import files

    print("Please upload your original marketing_data.csv file.")
    uploaded = files.upload()

    csv_files = [
        name for name in uploaded
        if name.lower().endswith(".csv")
    ]

    if not csv_files:
        raise FileNotFoundError(
            "No CSV file was uploaded. Please upload marketing_data.csv."
        )

    CSV_PATH = Path("/content") / csv_files[0]

print(f"Successfully located marketing dataset: {CSV_PATH.name}")


# PART 2: AGENT ROLE AND PROPOSED SYSTEM PROMPT


SYSTEM_PROMPT = """
You are a conservative marketing budget optimization assistant.

Your goal is to recommend daily advertising budget allocations
across Search, Social, and Display.

Use conversion rate (CVR = conversions / clicks) to evaluate
marketing channel performance.

Follow these rules:
1. Maintain a total daily advertising budget of $300.
2. Keep a minimum budget of $20 for every channel.
3. Never change a channel's budget by more than 20% per day.
4. Recommend shifting budget toward higher-performing channels.
5. Explain and log every budget decision.
6. Never modify live advertising accounts without authorization.
7. Use aggregate marketing data, not personal customer information.
8. Use factual, professional, brand-safe explanations.
9. Flag invalid or missing data for human review.
10. Never present simulated results as actual performance.
"""

CHANNELS = ("Search", "Social", "Display")

TOTAL_BUDGET = 300.0
MIN_BUDGET = 20.0
SHIFT_RATE = 0.15
MAX_DAILY_CHANGE = 0.20


# PART 3: READ AND VALIDATE MARKETING DATA


def read_marketing_data(csv_path):

    by_day = defaultdict(dict)

    with open(
        csv_path,
        "r",
        newline="",
        encoding="utf-8-sig"
    ) as file:

        # Supports ordinary CSV and tab-separated exports.
        sample = file.read(4096)
        file.seek(0)

        try:
            dialect = csv.Sniffer().sniff(
                sample,
                delimiters=",\t;"
            )
            delimiter = dialect.delimiter
        except csv.Error:
            delimiter = ","

        reader = csv.DictReader(file, delimiter=delimiter)

        required_columns = {
            "date",
            "channel",
            "spend",
            "impressions",
            "clicks",
            "conversions"
        }

        if not reader.fieldnames:
            raise ValueError("The uploaded file is empty.")

        reader.fieldnames = [
            name.strip()
            for name in reader.fieldnames
        ]

        if not required_columns.issubset(reader.fieldnames):
            raise ValueError(
                "Missing required CSV columns. Expected: "
                "date, channel, spend, impressions, clicks, conversions"
            )

        for row in reader:

            date = row["date"].strip()
            channel = row["channel"].strip()

            if channel not in CHANNELS:
                raise ValueError(
                    f"Unexpected marketing channel: {channel}"
                )

            if channel in by_day[date]:
                raise ValueError(
                    f"Duplicate channel {channel} on {date}"
                )

            # Handles values such as "5,126" and "5126".
            values = {
                key: float(
                    row[key].replace(",", "").strip()
                )
                for key in (
                    "spend",
                    "impressions",
                    "clicks",
                    "conversions"
                )
            }

            if (
                any(value < 0 for value in values.values())
                or values["spend"] <= 0
                or values["conversions"] > values["clicks"]
                or values["clicks"] > values["impressions"]
            ):
                raise ValueError(
                    f"Invalid marketing data: {row}"
                )

            by_day[date][channel] = values

    if not by_day:
        raise ValueError("No marketing records found.")

    for date, channels in by_day.items():

        if set(channels.keys()) != set(CHANNELS):
            raise ValueError(
                f"Missing marketing channel data on {date}"
            )

    history = dict(sorted(by_day.items()))

    print(f"Successfully loaded {len(history)} days of marketing data.")
    print(f"Total records: {len(history) * len(CHANNELS)}")
    print()

    return history


# PART 4: AGENT DECISION LOGIC


def recommend_budgets(previous_budgets, day_data):

    # Calculate conversion rate for each channel.
    cvr_scores = {}

    for channel in CHANNELS:

        clicks = day_data[channel]["clicks"]
        conversions = day_data[channel]["conversions"]

        if clicks > 0:
            cvr_scores[channel] = conversions / clicks
        else:
            cvr_scores[channel] = 0.0

    # Identify highest- and lowest-performing channels.
    best_channel = max(
        CHANNELS,
        key=lambda channel: cvr_scores[channel]
    )

    worst_channel = min(
        CHANNELS,
        key=lambda channel: cvr_scores[channel]
    )

    proposed_budgets = previous_budgets.copy()

    # Avoid shifting budgets when performance is tied.
    if (
        best_channel == worst_channel
        or abs(
            cvr_scores[best_channel] -
            cvr_scores[worst_channel]
        ) < 1e-12
    ):
        return (
            proposed_budgets,
            cvr_scores,
            "No change: channel conversion rates are tied."
        )

    # Target shift: 15% of the worst channel's current budget.
    target_shift = (
        previous_budgets[worst_channel] * SHIFT_RATE
    )

    # Guardrail: maximum donor reduction of 20%.
    donor_change_limit = (
        previous_budgets[worst_channel] *
        MAX_DAILY_CHANGE
    )

    # Guardrail: maximum recipient increase of 20%.
    recipient_change_limit = (
        previous_budgets[best_channel] *
        MAX_DAILY_CHANGE
    )

    # Guardrail: never reduce a channel below $20.
    available_budget = max(
        0.0,
        previous_budgets[worst_channel] - MIN_BUDGET
    )

    # Calculate the actual permitted budget shift.
    shift_amount = min(
        target_shift,
        donor_change_limit,
        recipient_change_limit,
        available_budget
    )

    if shift_amount > 1e-9:

        proposed_budgets[worst_channel] -= shift_amount
        proposed_budgets[best_channel] += shift_amount

        reason = (
            f"Shifted ${shift_amount:.2f} from "
            f"{worst_channel} to {best_channel}. "
            f"{best_channel} had the highest CVR "
            f"({cvr_scores[best_channel]:.2%}), "
            f"while {worst_channel} had the lowest "
            f"({cvr_scores[worst_channel]:.2%})."
        )

    else:

        reason = (
            f"No budget change: {worst_channel} "
            f"has reached the ${MIN_BUDGET:.2f} "
            f"minimum budget floor."
        )


    # SAFETY VALIDATION


    # Total budget must remain $300.
    assert abs(
        sum(proposed_budgets.values()) - TOTAL_BUDGET
    ) < 1e-7

    for channel in CHANNELS:

        # Every channel must retain at least $20.
        assert (
            proposed_budgets[channel] >= MIN_BUDGET - 1e-7
        )

        # Daily budget change cannot exceed 20%.
        budget_change = abs(
            proposed_budgets[channel] -
            previous_budgets[channel]
        )

        maximum_allowed = (
            previous_budgets[channel] *
            MAX_DAILY_CHANGE
        )

        assert budget_change <= maximum_allowed + 1e-7

    return proposed_budgets, cvr_scores, reason


# PART 5: EVALUATION FUNCTION


def estimate_conversions(budgets, next_day_data):

    """
    Estimate next-day conversions assuming conversions
    scale proportionally with advertising spend.

    IMPORTANT:
    This is a simulation, not observed performance.
    """

    estimated_total = 0.0

    for channel in CHANNELS:

        historical_spend = next_day_data[channel]["spend"]
        historical_conversions = next_day_data[channel][
            "conversions"
        ]

        estimated_channel_conversions = (
            historical_conversions *
            budgets[channel] /
            historical_spend
        )

        estimated_total += estimated_channel_conversions

    return estimated_total


# PART 6: RUN AUTONOMOUS OPTIMIZATION LOOP


def run_ad_optimization_agent():

    history = read_marketing_data(CSV_PATH)

    dates = list(history.keys())

    # Starting allocation: $100 per channel.
    current_budgets = {
        channel: TOTAL_BUDGET / len(CHANNELS)
        for channel in CHANNELS
    }

    decision_logs = []

    baseline_conversions = 0.0
    simulated_conversions = 0.0

    print("STARTING AD OPTIMIZATION AGENT")
    print("=" * 80)

    for day_index, date in enumerate(dates):

        print(
            f"\nDATE: {date} | DAY {day_index + 1}"
        )

        day_data = history[date]

        # Analyze performance and recommend new budgets.
        (
            current_budgets,
            cvr_scores,
            rationale
        ) = recommend_budgets(
            current_budgets,
            day_data
        )

        # Print conversion rates.
        print(
            "CVR PERFORMANCE: "
            + " | ".join(
                f"{channel}: {cvr_scores[channel]:.2%}"
                for channel in CHANNELS
            )
        )

        # Print decision explanation.
        print(f"AGENT DECISION: {rationale}")

        # Print updated budgets.
        print(
            "NEW BUDGET TARGETS: "
            + " | ".join(
                f"{channel}: ${current_budgets[channel]:.2f}"
                for channel in CHANNELS
            )
        )

        print("-" * 80)

        # Save daily decisions.
        decision_logs.append({

            "date": date,

            "search_budget": round(
                current_budgets["Search"], 2
            ),

            "social_budget": round(
                current_budgets["Social"], 2
            ),

            "display_budget": round(
                current_budgets["Display"], 2
            ),

            "search_cvr": round(
                cvr_scores["Search"], 6
            ),

            "social_cvr": round(
                cvr_scores["Social"], 6
            ),

            "display_cvr": round(
                cvr_scores["Display"], 6
            ),

            "rationale": rationale
        })

        # Evaluate today's allocation against the next day's
        # historical data, rather than the same day's data.
        if day_index + 1 < len(dates):

            next_date = dates[day_index + 1]
            next_day = history[next_date]

            baseline_conversions += sum(
                next_day[channel]["conversions"]
                for channel in CHANNELS
            )

            simulated_conversions += estimate_conversions(
                current_budgets,
                next_day
            )


    # PART 7: EXPORT DAILY DECISION LOG


    log_path = Path("/content/decision_log.csv")

    with open(
        log_path,
        "w",
        newline="",
        encoding="utf-8"
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=decision_logs[0].keys()
        )

        writer.writeheader()
        writer.writerows(decision_logs)

    print("\nDECISION LOG SAVED SUCCESSFULLY")
    print(f"File: {log_path}")


    # PART 8: EVALUATION RESULTS


    evaluation_days = len(dates) - 1

    total_spend = (
        evaluation_days * TOTAL_BUDGET
    )

    print("\n" + "=" * 80)
    print("FINAL AGENT EVALUATION RESULTS")
    print("=" * 80)

    print(f"Days processed: {len(dates)}")
    print(f"Days evaluated: {evaluation_days}")

    print(
        f"Total evaluated advertising budget: "
        f"${total_spend:,.2f}"
    )

    print(
        f"Baseline conversions: "
        f"{baseline_conversions:,.2f}"
    )

    print(
        f"Estimated agent conversions: "
        f"{simulated_conversions:,.2f}"
    )

    if baseline_conversions > 0:

        baseline_cpa = (
            total_spend / baseline_conversions
        )

        print(
            f"Baseline CPA: ${baseline_cpa:.2f}"
        )

    else:
        print("Baseline CPA: N/A")

    if simulated_conversions > 0:

        estimated_cpa = (
            total_spend / simulated_conversions
        )

        print(
            f"Estimated Agent CPA: ${estimated_cpa:.2f}"
        )

    else:
        print("Estimated Agent CPA: N/A")

    if baseline_conversions > 0:

        estimated_change = (
            (
                simulated_conversions /
                baseline_conversions
            ) - 1
        ) * 100

        print(
            f"Estimated Conversion Difference: "
            f"{estimated_change:+.1f}%"
        )

    print("\nFINAL RECOMMENDED BUDGET ALLOCATION")

    for channel in CHANNELS:

        print(
            f"{channel}: "
            f"${current_budgets[channel]:.2f}"
        )

    print("\nIMPORTANT EVALUATION LIMITATION:")
    print(
        "Estimated conversions assume linear scaling "
        "with advertising spend."
    )
    print(
        "These results do not demonstrate actual "
        "conversion improvements or real-world ROI."
    )

    print("\nAGENT EXECUTION COMPLETED SUCCESSFULLY")


# PART 9: EXECUTE THE AGENT


run_ad_optimization_agent()


Successfully located marketing dataset: marketing_data.csv
Successfully loaded 14 days of marketing data.
Total records: 42

STARTING AD OPTIMIZATION AGENT

DATE: 2026-10-01 | DAY 1
CVR PERFORMANCE: Search: 11.40% | Social: 4.07% | Display: 2.91%
AGENT DECISION: Shifted $15.00 from Display to Search. Search had the highest CVR (11.40%), while Display had the lowest (2.91%).
NEW BUDGET TARGETS: Search: $115.00 | Social: $100.00 | Display: $85.00
--------------------------------------------------------------------------------

DATE: 2026-10-02 | DAY 2
CVR PERFORMANCE: Search: 8.61% | Social: 3.21% | Display: 0.72%
AGENT DECISION: Shifted $12.75 from Display to Search. Search had the highest CVR (8.61%), while Display had the lowest (0.72%).
NEW BUDGET TARGETS: Search: $127.75 | Social: $100.00 | Display: $72.25
--------------------------------------------------------------------------------

DATE: 2026-10-03 | DAY 3
CVR PERFORMANCE: Search: 11.73% | Social: 7.07% | Display: 2.88%
AGENT D